# Loan Prediction — Intelex vs CuPy benchmark

Dataset: https://www.kaggle.com/datasets/ethicalstar/loan-prediction/code

Models: LightGBM, XGBoost, Logistic Regression, Random Forest, AdaBoost, MLP.

Sections are grouped by implementation because `patch_sklearn()` patches the whole kernel: run **BareBones** and **CuPy** first, then the single patch cell, then **Intelex**.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
import warnings
import cupy as cp
from sklearnex import patch_sklearn
import numpy as np
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neural_network import MLPClassifier


## BareBones (plain scikit-learn)

### LightGBM

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def clean_column_names(data):
    # Remove special characters and spaces from column names
    data.columns = data.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
    print("\nCleaned Column Names:")
    return data

def mainBareBones():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with your dataset's path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Clean column names to remove unsupported characters
    print("\nCleaning column names...")
    data = clean_column_names(data)

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married_Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    categorical_columns = [col for col in categorical_columns if col in data.columns]  # Ensure columns exist
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Clean column names after encoding to remove any unsupported characters
    print("\nCleaning encoded column names...")
    data.columns = data.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'], errors='ignore')
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Initialize the LightGBM Classifier
    print("\nInitializing the LightGBM classifier...")
    lgbm = LGBMClassifier(
        objective='binary', 
        random_state=42, 
        n_estimators=50
    )

    # Train the model
    print("\nTraining the LightGBM model...")
    lgbm.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = lgbm.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainBareBones()


Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning column names...

Cleaned Column Names:

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0

Optimizing datatypes...
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Cleaning encoded column names...

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Initializing the LightGBM classifier...

Training the LightGBM model...
[LightGBM] [Info] Number of positive: 24743, number of negative: 176857
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004866 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total 

### XGBoost

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainBareBones():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with your dataset's path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes to reduce memory usage
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Convert categorical variables to numeric with one-hot encoding
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Sanitize column names for compatibility with XGBoost
    print("\nSanitizing column names for XGBoost compatibility...")
    X.columns = [col.replace('[', '').replace(']', '').replace('<', '').replace('>', '').replace(' ', '_') for col in X.columns]

    # Convert data to optimized types
    print("\nConverting data to optimized types...")
    X = X.astype(np.float32)
    y = y.astype(np.int32)

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Convert to DMatrix
    print("\nConverting data to DMatrix format for XGBoost...")
    train_dmatrix = xgb.DMatrix(X_train, label=y_train)
    test_dmatrix = xgb.DMatrix(X_test, label=y_test)

    # Parameters for XGBoost
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for GPU
        'device': 'cuda:0', # Use GPU,
        'random_state': 42 
    }

    # Train the XGBoost model
    print("\nTraining the XGBoost model...")
    model = xgb.train(params, train_dmatrix, num_boost_round=100)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(test_dmatrix)
    y_pred_binary = [1 if pred > 0.5 else 0 for pred in y_pred]

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred_binary)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_binary))

if __name__ == "__main__":
    mainBareBones()


Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0

Optimizing datatypes...
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Sanitizing column names for XGBoost compatibility...

Converting data to optimized types...

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Converting data to DMatrix format for XGBoost...

Training the XGBoost model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 88.06%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94     44147
           1       0.68      0.07      0.13      6253

    accur

### Logistic Regression

In [ ]:
def optimize_data_types(data):
    for col in data.columns:
        if data[col].dtype == 'object':  # Convert categorical text data to 'category'
            data[col] = data[col].astype('category')
        elif data[col].dtype in ['int64', 'float64']:
            # Downcast numerical data types
            if data[col].dtype == 'int64':
                if data[col].min() >= np.iinfo(np.int8).min and data[col].max() <= np.iinfo(np.int8).max:
                    data[col] = data[col].astype(np.int8)
                elif data[col].min() >= np.iinfo(np.int16).min and data[col].max() <= np.iinfo(np.int16).max:
                    data[col] = data[col].astype(np.int16)
                elif data[col].min() >= np.iinfo(np.int32).min and data[col].max() <= np.iinfo(np.int32).max:
                    data[col] = data[col].astype(np.int32)
            elif data[col].dtype == 'float64':
                if data[col].min() >= np.finfo(np.float32).min and data[col].max() <= np.finfo(np.float32).max:
                    data[col] = data[col].astype(np.float32)
    return data

def mainBareBones():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with the path to your dataset
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Optimize data types
    data = optimize_data_types(data)

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype.name == 'category':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train logistic regression model
    print("\nTraining the Logistic Regression model...")
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=1))

if __name__ == "__main__":
    mainBareBones()

Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Training the Logistic Regression model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 87.59%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.93     44147
           1       1.00      0.00      0.00      6253

    accuracy                           0.88     50400
   macro avg       0.94      0.50      0.47     50400
weighted avg       0.89      0.88      0.82     50400



### Random Forest

In [ ]:
def mainBareBones():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with the path to your dataset
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train Random Forest model
    print("\nTraining the Random Forest classifier...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainBareBones()


Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Training the Random Forest classifier...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 89.85%

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.94     44147
           1       0.60      0.54      0.57      6253

    accuracy                           0.90     50400
   macro avg       0.77      0.74      0.75     50400
weighted avg       0.89      0.90      0.90     50400



### AdaBoost

In [ ]:
def optimize_data_types(data):
    # Convert categorical columns to 'category' type
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    for col in categorical_columns:
        if col in data.columns:  # Check if the column exists
            data[col] = data[col].astype('category')
    
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include='number').columns:
        data[col] = pd.to_numeric(data[col], downcast='integer')
    
    return data

def mainBareBones():
    # Load data
    data = pd.read_csv('Loan Prediction.csv')

    # Drop rows with missing values
    data = data.dropna()

    # Optimize data types
    data = optimize_data_types(data)

    # Encode categorical data
    encoder = LabelEncoder()
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    for col in categorical_columns:
        if col in data.columns:  # Check if the column exists
            data[col] = encoder.fit_transform(data[col])

    # Split features (X) and target variable (y)
    target_column = 'Risk_Flag'  # Target variable
    X = data.drop(columns=[target_column])
    y = data[target_column]

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initialize and train AdaBoost Classifier
    model = AdaBoostClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate accuracy
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=1))

if __name__ == "__main__":
    mainBareBones()


Evaluating the model...
Accuracy: 87.59%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.93     44147
           1       1.00      0.00      0.00      6253

    accuracy                           0.88     50400
   macro avg       0.94      0.50      0.47     50400
weighted avg       0.89      0.88      0.82     50400



### MLP

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainBareBones():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with your dataset's path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes to reduce memory usage
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Convert categorical variables to numeric with one-hot encoding
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Convert data to optimized types
    print("\nConverting data to optimized types...")
    X = X.astype(np.float32)
    y = y.astype(np.int32)

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the model using MLPClassifier
    print("\nTraining the MLP model...")
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

if __name__ == "__main__":
    mainBareBones()

Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0

Optimizing datatypes...
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Converting data to optimized types...

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Training the MLP model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 87.59%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.93     44147
           1       0.00      0.00      0.00      6253

    accuracy                           0.88     50400
   macro avg       0.44      0.50      0.47     50400
weighted 

## CuPy

### LightGBM

In [ ]:
import cupy as cp
import pandas as pd

def optimize_datatypes(data):
    # Downcast numerical columns
    for col in data.select_dtypes(include=['int']):
        data[col] = cp.asnumpy(cp.array(data[col], dtype=cp.int32))
    for col in data.select_dtypes(include=['float']):
        data[col] = cp.asnumpy(cp.array(data[col], dtype=cp.float32))
    return data

def clean_column_names(data):
    # Remove special characters and spaces from column names
    data.columns = data.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
    print("\nCleaned Column Names:")
    return data

def mainCuPY():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with your dataset's path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Clean column names to remove unsupported characters
    print("\nCleaning column names...")
    data = clean_column_names(data)

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married_Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    categorical_columns = [col for col in categorical_columns if col in data.columns]  # Ensure columns exist
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Clean column names after encoding to remove any unsupported characters
    print("\nCleaning encoded column names...")
    data.columns = data.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'], errors='ignore')
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Initialize the LightGBM Classifier
    print("\nInitializing the LightGBM classifier...")
    lgbm = LGBMClassifier(
        objective='binary', 
        random_state=42, 
        n_estimators=50
    )

    # Train the model
    print("\nTraining the LightGBM model...")
    lgbm.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = lgbm.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainCuPY()


Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning column names...

Cleaned Column Names:

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0

Optimizing datatypes...
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Cleaning encoded column names...

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Initializing the LightGBM classifier...

Training the LightGBM model...
[LightGBM] [Info] Number of positive: 24743, number of negative: 176857
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005033 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total 

### XGBoost

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainCuPY():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with the path to your dataset
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")

    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes to reduce memory usage
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Sanitize column names for compatibility with XGBoost
    print("\nSanitizing column names for XGBoost compatibility...")
    X.columns = [col.replace('[', '').replace(']', '').replace('<', '').replace('>', '').replace(' ', '_') for col in X.columns]

    # Convert features and target to optimized numeric types
    print("\nConverting data to optimized types...")
    X = X.astype(np.float32)
    y = y.astype(np.int32)

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Convert data to CuPy arrays
    print("\nConverting data to CuPy arrays...")
    X_train_cp = cp.asarray(X_train)
    X_test_cp = cp.asarray(X_test)
    y_train_cp = cp.asarray(y_train)
    y_test_cp = cp.asarray(y_test)

    # Convert CuPy arrays to DMatrix for XGBoost
    print("\nConverting data to DMatrix format for XGBoost...")
    train_dmatrix = xgb.DMatrix(X_train_cp, label=cp.asnumpy(y_train_cp))
    test_dmatrix = xgb.DMatrix(X_test_cp, label=cp.asnumpy(y_test_cp))

    # Parameters for XGBoost
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for GPU
        'device': 'cuda:0', # Use GPU,
        'random_state': 42 
    }

    # Train the XGBoost model
    print("\nTraining the XGBoost model...")
    model = xgb.train(params, train_dmatrix, num_boost_round=100)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(test_dmatrix)
    y_pred_binary = [1 if pred > 0.5 else 0 for pred in y_pred]

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(cp.asnumpy(y_test_cp), y_pred_binary)
    print(f"Accuracy (CuPy Optimized): {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(cp.asnumpy(y_test_cp), y_pred_binary))

if __name__ == "__main__":
    mainCuPY()

Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0

Optimizing datatypes...
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Sanitizing column names for XGBoost compatibility...

Converting data to optimized types...

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Converting data to CuPy arrays...

Converting data to DMatrix format for XGBoost...

Training the XGBoost model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy (CuPy Optimized): 88.06%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94     44147
           1

### Logistic Regression

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

def mainCuPY():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with the path to your dataset
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Convert to NumPy arrays before converting to CuPy arrays
    print("\nConverting DataFrame to arrays...")
    X = X.values  # DataFrame to NumPy
    y = y.values

    # Ensure the data contains no invalid values (inf, -inf, NaN)
    print("\nCleaning invalid values in the dataset...")
    X = pd.DataFrame(X).replace([np.inf, -np.inf], np.nan).dropna().values
    
    # Ensure all columns in X are numeric and cast to float
    if not np.issubdtype(X.dtype, np.number):
        X = pd.DataFrame(X).apply(pd.to_numeric, errors='coerce').fillna(0).values
    
    # Explicitly cast X and y to float32
    X = X.astype(np.float32)  
    y = y.astype(np.float32)  
    
    # Convert NumPy arrays to CuPy arrays
    print("\nTransferring data to GPU...")
    X = cp.asarray(X)
    y = cp.asarray(y)
    print("Data successfully transferred to GPU!")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(cp.asnumpy(X), cp.asnumpy(y), test_size=0.2, random_state=42)
    X_train = cp.asarray(X_train)
    X_test = cp.asarray(X_test)
    y_train = cp.asarray(y_train)
    y_test = cp.asarray(y_test)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train logistic regression model
    print("\nTraining the Logistic Regression model...")
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(cp.asnumpy(X_train), cp.asnumpy(y_train))  # Use `.asnumpy()` to transfer data back to CPU for sklearn
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = cp.asarray(model.predict(cp.asnumpy(X_test)))  # Use `.asnumpy()` to transfer data to CPU, then back to GPU

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(cp.asnumpy(y_test), cp.asnumpy(y_pred))  # Use `.asnumpy()` for sklearn functions
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(cp.asnumpy(y_test), cp.asnumpy(y_pred), zero_division=1))

if __name__ == "__main__":
    mainCuPY()


Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Converting DataFrame to arrays...

Cleaning invalid values in the dataset...

Transferring data to GPU...
Data successfully transferred to GPU!

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Training the Logistic Regression model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 87.59%

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      1.00      0.93     44147
         1.0       1.00      0.00      0.00      6253

    accuracy        

### Random Forest

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

def mainCuPY():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with the path to your dataset
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Convert to NumPy arrays before converting to CuPy arrays
    print("\nConverting DataFrame to arrays...")
    X = X.values  # DataFrame to NumPy
    y = y.values

    # Ensure the data contains no invalid values (inf, -inf, NaN)
    print("\nCleaning invalid values in the dataset...")
    X = pd.DataFrame(X).replace([np.inf, -np.inf], np.nan).dropna().values
    
    # Ensure all columns in X are numeric and cast to float
    if not np.issubdtype(X.dtype, np.number):
        X = pd.DataFrame(X).apply(pd.to_numeric, errors='coerce').fillna(0).values
    
    # Explicitly cast X and y to float32
    X = X.astype(np.float32)  
    y = y.astype(np.float32)  
    
    # Convert NumPy arrays to CuPy arrays
    print("\nTransferring data to GPU...")
    X = cp.asarray(X)
    y = cp.asarray(y)
    print("Data successfully transferred to GPU!")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(cp.asnumpy(X), cp.asnumpy(y), test_size=0.2, random_state=42)
    X_train = cp.asarray(X_train)
    X_test = cp.asarray(X_test)
    y_train = cp.asarray(y_train)
    y_test = cp.asarray(y_test)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train Random Forest model
    print("\nTraining the Random Forest classifier...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(cp.asnumpy(X_train), cp.asnumpy(y_train))  # Use `.asnumpy()` to transfer data back to CPU for sklearn
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = cp.asarray(model.predict(cp.asnumpy(X_test)))  # Use `.asnumpy()` to transfer data to CPU, then back to GPU

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(cp.asnumpy(y_test), cp.asnumpy(y_pred))  # Use `.asnumpy()` for sklearn functions
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(cp.asnumpy(y_test), cp.asnumpy(y_pred), zero_division=1))

if __name__ == "__main__":
    mainCuPY()

Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Converting DataFrame to arrays...

Cleaning invalid values in the dataset...

Transferring data to GPU...
Data successfully transferred to GPU!

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Training the Random Forest classifier...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 89.85%

Classification Report:
              precision    recall  f1-score   support

         0.0       0.94      0.95      0.94     44147
         1.0       0.60      0.54      0.57      6253

    accuracy         

### AdaBoost

In [ ]:
def optimize_data_types(data):
    # Convert categorical columns to 'category' type
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    for col in categorical_columns:
        if col in data.columns:  # Check if the column exists
            data[col] = data[col].astype('category')
    
    # Downcast numerical columns to reduce memory usage with CuPy
    for col in data.select_dtypes(include='number').columns:
        cupy_array = cp.array(data[col], dtype=cp.float32)
        data[col] = cupy_array.get()
    
    return data

def mainCuPY():
    # Load data into a pandas DataFrame
    data = pd.read_csv('Loan Prediction.csv')

    # Drop rows with missing values
    data = data.dropna()

    # Optimize data types
    data = optimize_data_types(data)

    # Encode categorical data
    encoder = LabelEncoder()
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    for col in categorical_columns:
        if col in data.columns:  # Check if the column exists
            data[col] = encoder.fit_transform(data[col])

    # Convert features and target to CuPy arrays
    target_column = 'Risk_Flag'  # Target variable
    X = cp.array(data.drop(columns=[target_column]))
    y = cp.array(data[target_column])

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initialize and train AdaBoost Classifier
    model = AdaBoostClassifier(n_estimators=50, random_state=42)
    model.fit(cp.asnumpy(X_train), cp.asnumpy(y_train))  # Convert to NumPy for compatibility

    # Make predictions
    y_pred = model.predict(cp.asnumpy(X_test))  # Convert to NumPy for compatibility

    # Evaluate accuracy
    print("\nEvaluating the model...")
    accuracy = accuracy_score(cp.asnumpy(y_test), y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(cp.asnumpy(y_test), y_pred, zero_division=1))

if __name__ == "__main__":
    mainCuPY()



Evaluating the model...
Accuracy: 87.59%

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      1.00      0.93     44147
         1.0       1.00      0.00      0.00      6253

    accuracy                           0.88     50400
   macro avg       0.94      0.50      0.47     50400
weighted avg       0.89      0.88      0.82     50400



### MLP

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns
    for col in data.select_dtypes(include=['int']):
        data[col] = cp.array(data[col], dtype=cp.int32).get()
    for col in data.select_dtypes(include=['float']):
        data[col] = cp.array(data[col], dtype=cp.float32).get()
    return data

def mainCuPY():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with your dataset's path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Convert categorical variables to numeric with one-hot encoding
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = cp.array(data.drop(columns=['Risk_Flag', 'Id']).values, dtype=cp.float32).get()  # Convert to CuPy
    y = cp.array(data['Risk_Flag'].values, dtype=cp.int32).get()
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the model using MLPClassifier
    print("\nTraining the MLP model...")
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

if __name__ == "__main__":
    mainCuPY()


Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0

Optimizing datatypes...
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Training the MLP model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 87.59%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.93     44147
           1       0.00      0.00      0.00      6253

    accuracy                           0.88     50400
   macro avg       0.44      0.50      0.47     50400
weighted avg       0.77      0.88      0.82     

## Intelex (scikit-learn-intelex)

Apply the Intel patch once; everything below runs patched.

In [ ]:
patch_sklearn()  # Apply Intel optimizations


Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


### LightGBM

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def clean_column_names(data):
    # Remove special characters and spaces from column names
    data.columns = data.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
    print("\nCleaned Column Names:")
    return data

def mainIntelex():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with your dataset's path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Clean column names to remove unsupported characters
    print("\nCleaning column names...")
    data = clean_column_names(data)

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married_Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    categorical_columns = [col for col in categorical_columns if col in data.columns]  # Ensure columns exist
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Clean column names after encoding to remove any unsupported characters
    print("\nCleaning encoded column names...")
    data.columns = data.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'], errors='ignore')
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Initialize the LightGBM Classifier
    print("\nInitializing the LightGBM classifier...")
    lgbm = LGBMClassifier(
        objective='binary', 
        random_state=42, 
        n_estimators=50
    )

    # Train the model
    print("\nTraining the LightGBM model...")
    lgbm.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = lgbm.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainIntelex()

Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning column names...

Cleaned Column Names:

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0

Optimizing datatypes...
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Cleaning encoded column names...

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Initializing the LightGBM classifier...

Training the LightGBM model...
[LightGBM] [Info] Number of positive: 24743, number of negative: 176857
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004346 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total 

### XGBoost

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainIntelex():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with your dataset's path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes to reduce memory usage
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Convert categorical variables to numeric with one-hot encoding
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Sanitize column names for compatibility with XGBoost
    print("\nSanitizing column names for XGBoost compatibility...")
    X.columns = [col.replace('[', '').replace(']', '').replace('<', '').replace('>', '').replace(' ', '_') for col in X.columns]

    # Convert data to optimized types
    print("\nConverting data to optimized types...")
    X = X.astype(np.float32)
    y = y.astype(np.int32)

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Convert to DMatrix
    print("\nConverting data to DMatrix format for XGBoost...")
    train_dmatrix = xgb.DMatrix(X_train, label=y_train)
    test_dmatrix = xgb.DMatrix(X_test, label=y_test)

    # Parameters for XGBoost
    params = {
        'objective': 'binary:logistic',
        'tree_method': 'hist',  # Optimized for GPU
        'device': 'cuda:0', # Use GPU,
        'random_state': 42 
    }

    # Train the XGBoost model
    print("\nTraining the XGBoost model...")
    model = xgb.train(params, train_dmatrix, num_boost_round=100)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(test_dmatrix)
    y_pred_binary = [1 if pred > 0.5 else 0 for pred in y_pred]

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred_binary)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_binary))

if __name__ == "__main__":
    mainIntelex()

Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0

Optimizing datatypes...
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Sanitizing column names for XGBoost compatibility...

Converting data to optimized types...

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Converting data to DMatrix format for XGBoost...

Training the XGBoost model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 88.06%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.94     44147
           1       0.68      0.07      0.13      6253

    accur

### Logistic Regression

In [ ]:
def optimize_data_types(data):
    for col in data.columns:
        if data[col].dtype == 'object':  # Convert categorical text data to 'category'
            data[col] = data[col].astype('category')
        elif data[col].dtype in ['int64', 'float64']:
            # Use smaller data types where possible
            if data[col].dtype == 'int64':
                if data[col].min() >= np.iinfo(np.int8).min and data[col].max() <= np.iinfo(np.int8).max:
                    data[col] = data[col].astype(np.int8)
                elif data[col].min() >= np.iinfo(np.int16).min and data[col].max() <= np.iinfo(np.int16).max:
                    data[col] = data[col].astype(np.int16)
                elif data[col].min() >= np.iinfo(np.int32).min and data[col].max() <= np.iinfo(np.int32).max:
                    data[col] = data[col].astype(np.int32)
            elif data[col].dtype == 'float64':
                if data[col].min() >= np.finfo(np.float32).min and data[col].max() <= np.finfo(np.float32).max:
                    data[col] = data[col].astype(np.float32)
    return data

def mainIntelex():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with the path to your dataset
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Optimize data types
    data = optimize_data_types(data)

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")

    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype.name == 'category':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())

    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train logistic regression model
    print("\nTraining the Logistic Regression model...")
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=1))

if __name__ == "__main__":
    mainIntelex()


Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Training the Logistic Regression model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 87.59%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.93     44147
           1       1.00      0.00      0.00      6253

    accuracy                           0.88     50400
   macro avg       0.94      0.50      0.47     50400
weighted avg       0.89      0.88      0.82     50400



### Random Forest

In [ ]:
def mainIntelex():
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with the path to your dataset
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Convert categorical variables to numeric
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train Random Forest model
    print("\nTraining the Random Forest classifier...")
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

if __name__ == "__main__":
    mainIntelex()


Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Training the Random Forest classifier...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 89.85%

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.94     44147
           1       0.60      0.54      0.57      6253

    accuracy                           0.90     50400
   macro avg       0.77      0.74      0.75     50400
weighted avg       0.89      0.90      0.90     50400



### AdaBoost

In [ ]:
def optimize_data_types(data):
    # Convert categorical columns to 'category' type
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    for col in categorical_columns:
        if col in data.columns:  # Check if the column exists
            data[col] = data[col].astype('category')
    
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include='number').columns:
        data[col] = pd.to_numeric(data[col], downcast='integer')
    
    return data

def mainIntelex():
    # Load data
    data = pd.read_csv('Loan Prediction.csv')

    # Drop rows with missing values
    data = data.dropna()

    # Optimize data types
    data = optimize_data_types(data)

    # Encode categorical data
    encoder = LabelEncoder()
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    for col in categorical_columns:
        if col in data.columns:  # Check if the column exists
            data[col] = encoder.fit_transform(data[col])

    # Split features (X) and target variable (y)
    target_column = 'Risk_Flag'  # Target variable
    X = data.drop(columns=[target_column])
    y = data[target_column]

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initialize and train AdaBoost Classifier
    model = AdaBoostClassifier(n_estimators=50, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate accuracy
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=1))

if __name__ == "__main__":
    mainIntelex()


Evaluating the model...
Accuracy: 87.59%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.93     44147
           1       1.00      0.00      0.00      6253

    accuracy                           0.88     50400
   macro avg       0.94      0.50      0.47     50400
weighted avg       0.89      0.88      0.82     50400



### MLP

In [ ]:
def optimize_datatypes(data):
    # Downcast numerical columns to reduce memory usage
    for col in data.select_dtypes(include=['int']):
        data[col] = pd.to_numeric(data[col], downcast='integer')
    for col in data.select_dtypes(include=['float']):
        data[col] = pd.to_numeric(data[col], downcast='float')
    return data

def mainIntelex():
    # Suppress warnings
    warnings.filterwarnings("ignore", category=FutureWarning)
    
    # Load the dataset
    file_path = "Loan Prediction.csv"  # Replace with your dataset's path
    print("Loading dataset...")
    try:
        data = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {data.shape}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        exit()

    # Data Cleaning
    print("\nCleaning and preprocessing data...")
    missing_values_before = data.isnull().sum().sum()
    print(f"Missing values before cleaning: {missing_values_before}")
    
    # Fill missing values for categorical and numerical columns
    for col in data.columns:
        if data[col].dtype == 'object':  # Categorical columns
            data[col] = data[col].fillna(data[col].mode()[0])
        else:  # Numerical columns
            data[col] = data[col].fillna(data[col].median())
    
    missing_values_after = data.isnull().sum().sum()
    print(f"Missing values after cleaning: {missing_values_after}")

    # Optimize datatypes to reduce memory usage
    print("\nOptimizing datatypes...")
    data = optimize_datatypes(data)

    # Convert categorical variables to numeric with one-hot encoding
    print("Converting categorical variables to numeric with one-hot encoding...")
    categorical_columns = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']
    data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)
    print(f"Data shape after encoding: {data.shape}")

    # Feature-target split
    print("\nSeparating features and target variable...")
    X = data.drop(columns=['Risk_Flag', 'Id'])  # Dropping Id and target column
    y = data['Risk_Flag']
    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Convert data to optimized types
    print("\nConverting data to optimized types...")
    X = X.astype(np.float32)
    y = y.astype(np.int32)

    # Train-test split
    print("\nSplitting data into training and testing sets...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

    # Train the model using MLPClassifier
    print("\nTraining the MLP model...")
    model = MLPClassifier(
        hidden_layer_sizes=(50, 30),  
        max_iter=1200,               
        learning_rate='adaptive',            
        early_stopping=True,        
        random_state=42              
    )
    model.fit(X_train, y_train)
    print("Model training completed!")

    # Make predictions
    print("\nMaking predictions on the test set...")
    y_pred = model.predict(X_test)

    # Evaluate the model
    print("\nEvaluating the model...")
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

if __name__ == "__main__":
    mainIntelex()


Loading dataset...
Dataset loaded successfully! Shape: (252000, 13)

Cleaning and preprocessing data...
Missing values before cleaning: 0
Missing values after cleaning: 0

Optimizing datatypes...
Converting categorical variables to numeric with one-hot encoding...
Data shape after encoding: (252000, 405)

Separating features and target variable...
Features shape: (252000, 403), Target shape: (252000,)

Converting data to optimized types...

Splitting data into training and testing sets...
Training set shape: (201600, 403), Testing set shape: (50400, 403)

Training the MLP model...
Model training completed!

Making predictions on the test set...

Evaluating the model...
Accuracy: 87.59%

Classification Report:
              precision    recall  f1-score   support

           0       0.88      1.00      0.93     44147
           1       0.00      0.00      0.00      6253

    accuracy                           0.88     50400
   macro avg       0.44      0.50      0.47     50400
weighted 